# Análise de custo dos alertas — FD001

A etapa anterior mostrou que o limiar de alerta cria um compromisso entre antecipação e risco. Agora esse compromisso será traduzido em cenários de custo. O objetivo não será afirmar um valor financeiro universal, pois os custos reais dependem da operação; será mostrar como uma mudança nessas prioridades altera o limiar recomendado.

Será mantido o mesmo Extra Trees e as mesmas previsões fora da amostra utilizadas no early warning. Assim, a comparação será feita sobre as consequências dos alertas, e não sobre uma nova escolha de modelo.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import plotly.express as px
import plotly.io as pio
from multilang import LangMap

pio.renderers.default = 'vscode'
lang = LangMap(source='pt', target='pt')

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'data' / 'raw' / 'dataset').exists():
    PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from cmapss_eda import load_split
from cmapss_modeling import add_rul_label, candidate_rul_features, nonconstant_features
from cmapss_optuna import build_pipeline_from_params, optimize_model
from cmapss_early_warning import generate_oof_rul_predictions
from cmapss_costs import evaluate_cost_grid

DATA_DIR = PROJECT_ROOT / 'data' / 'raw' / 'dataset'

## 1. Reutilização das previsões fora da amostra

O Extra Trees combina várias árvores com aleatoriedade adicional na construção das regras (Geurts et al., 2006). O modelo será reconstruído a partir do melhor trial v1, e a separação por motor será preservada na geração das previsões fora da amostra (Pedregosa et al., 2011).

In [2]:
train = add_rul_label(load_split(DATA_DIR, 'train', 'FD001'))
features = nonconstant_features(
    train,
    candidate_rul_features(include_settings=True),
)
study = optimize_model(
    train,
    features=features,
    model_name='Extra Trees',
    n_trials=100,
    n_splits=5,
    random_state=42,
    storage_dir=PROJECT_ROOT / 'outputs' / 'optuna',
    search_version='v1',
)
model = build_pipeline_from_params(
    model_name='Extra Trees',
    params=study.best_params,
    n_features=len(features),
    random_state=42,
    search_version='v1',
)
oof_predictions = generate_oof_rul_predictions(
    train, features=features, model=model, n_splits=5,
)

[I 2026-09-26 16:41:45,827] Using an existing study with `study_name='fd001_extra_trees_v1'` instead of creating a new one.


## 2. Cenários de custo

Os valores abaixo são unidades relativas, não valores monetários reais. Em todos os cenários, uma manutenção planejada após um alerta oportuno recebe custo 1. O alerta prematuro recebe custo maior porque antecipa a intervenção, e a falha não detectada recebe o maior custo por representar a perda da oportunidade de manutenção.

O cenário intermediário será usado apenas como referência visual. A decisão operacional real deverá substituir esses valores por custos da organização.

In [3]:
scenarios = {
    'Intermediário': {
        'alerta_oportuno': 1,
        'alerta_prematuro': 3,
        'falha_nao_detectada': 10,
    },
    'Falha muito cara': {
        'alerta_oportuno': 1,
        'alerta_prematuro': 2,
        'falha_nao_detectada': 20,
    },
    'Intervenção prematura cara': {
        'alerta_oportuno': 1,
        'alerta_prematuro': 8,
        'falha_nao_detectada': 10,
    },
}

cost_summary = evaluate_cost_grid(
    oof_predictions,
    thresholds=[10, 20, 30, 40, 50],
    warning_horizon=30,
    confirmation_cycles=3,
    scenarios=scenarios,
)
cost_summary.round(2)

,cenario,limiar_alerta,horizonte_alerta,ciclos_confirmacao,alertas_oportunos,alertas_prematuros,falhas_nao_detectadas,custo_alerta_oportuno,custo_alerta_prematuro,custo_falha_nao_detectada,custo_total,custo_medio_motor
0,Intermediário,10,30,3,97,0,3,1,3,10,127.0,1.27
1,Falha muito cara,10,30,3,97,0,3,1,2,20,157.0,1.57
2,Intervenção prematura cara,10,30,3,97,0,3,1,8,10,127.0,1.27
3,Intermediário,20,30,3,99,1,0,1,3,10,102.0,1.02
4,Falha muito cara,20,30,3,99,1,0,1,2,20,101.0,1.01
5,Intervenção prematura cara,20,30,3,99,1,0,1,8,10,107.0,1.07
6,Intermediário,30,30,3,86,14,0,1,3,10,128.0,1.28
7,Falha muito cara,30,30,3,86,14,0,1,2,20,114.0,1.14
8,Intervenção prematura cara,30,30,3,86,14,0,1,8,10,198.0,1.98
9,Intermediário,40,30,3,54,46,0,1,3,10,192.0,1.92


### Insights

A tabela mostra que o melhor limiar depende do custo atribuído a cada consequência. Quando a falha não detectada se torna muito cara, uma política que aceita mais alertas prematuros pode passar a ser preferível. Quando a intervenção antecipada é muito cara, o modelo tende a favorecer um limiar mais conservador. Esses valores ainda são cenários didáticos e não devem ser apresentados como custo industrial observado.

In [4]:
best_by_scenario = cost_summary.loc[
    cost_summary.groupby('cenario')['custo_medio_motor'].idxmin()
].sort_values('cenario')
best_by_scenario[['cenario', 'limiar_alerta', 'custo_medio_motor', 'alertas_oportunos', 'alertas_prematuros', 'falhas_nao_detectadas']].round(2)

,cenario,limiar_alerta,custo_medio_motor,alertas_oportunos,alertas_prematuros,falhas_nao_detectadas
4,Falha muito cara,20,1.01,99,1,0
3,Intermediário,20,1.02,99,1,0
5,Intervenção prematura cara,20,1.07,99,1,0


In [5]:
fig = px.line(
    cost_summary,
    x='limiar_alerta',
    y='custo_medio_motor',
    color='cenario',
    markers=True,
    labels={
        'limiar_alerta': 'Limiar de alerta (RUL em ciclos)',
        'custo_medio_motor': 'Custo médio relativo por motor',
        'cenario': 'Cenário',
    },
    title=lang({'title': 'Custo relativo por limiar e cenário'})['title'],
    height=550,
)
fig.show()

### Insights

A figura transforma a escolha do limiar em uma decisão transparente. Não será escolhido um valor apenas porque produziu a menor taxa de erro; será necessário declarar qual consequência está sendo priorizada. Em uma aplicação real, os custos deverão ser revisados com especialistas de manutenção, operação e segurança antes de qualquer uso do modelo.

## Referências

GEURTS, P.; ERNST, D.; WEHENKEL, L. Extremely randomized trees. Machine Learning, v. 63, p. 3–42, 2006.

PEDREGOSA, F. et al. Scikit-learn: Machine Learning in Python. Journal of Machine Learning Research, v. 12, p. 2825–2830, 2011.